# LLVIP Dataset Audit
Programmatic verification of RGB/thermal pairing, image quality, annotations, and data integrity.

In [ ]:
import sys
from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import csv
import io
import os
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

sys.path.insert(0, str(Path('..').resolve()))
from src.config import (
    DATA_VISIBLE_ROOT, DATA_VISIBLE_TRAIN, DATA_VISIBLE_TEST,
    DATA_INFRARED_ROOT, DATA_INFRARED_TRAIN, DATA_INFRARED_TEST,
    DATA_ANNOTATIONS,
    NUM_VISIBLE_TRAIN, NUM_VISIBLE_TEST,
    NUM_INFRARED_TRAIN, NUM_INFRARED_TEST,
    NUM_TOTAL_PAIRS, NUM_ANNOTATIONS,
)
from src.data.pairing import create_pair_records, PairReport
from src.data.dataset_loader import get_image_info
from src.data.preprocessing import parse_voc_annotation

## 1. Dataset Overview

In [ ]:
print('='*60)
print('LLVIP DATASET AUDIT')
print('='*60)
print(f'Dataset root: {DATA_VISIBLE_ROOT}')
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()
print('Directory Structure:')
base = DATA_VISIBLE_ROOT.parent
for d in sorted(base.iterdir()):
    if d.is_dir():
        for dd in sorted(d.iterdir()):
            if dd.is_dir():
                n = len([f for f in dd.rglob('*') if f.is_file()])
                print(f'  {dd.relative_to(base)}/ -- {n} files')
            else:
                print(f'  {dd.relative_to(base)} -- {dd.stat().st_size} bytes')
    else:
        print(f'  {d.name} -- {d.stat().st_size} bytes')

## 2. File Count Verification

In [ ]:
rgb_train = len([f for f in DATA_VISIBLE_TRAIN.rglob('*') if f.is_file()])
rgb_test = len([f for f in DATA_VISIBLE_TEST.rglob('*') if f.is_file()])
rgb_total = rgb_train + rgb_test
thermal_train = len([f for f in DATA_INFRARED_TRAIN.rglob('*') if f.is_file()])
thermal_test = len([f for f in DATA_INFRARED_TEST.rglob('*') if f.is_file()])
thermal_total = thermal_train + thermal_test
ann_count = len([f for f in DATA_ANNOTATIONS.rglob('*') if f.is_file()])

data = {
    'Metric': ['RGB images (train)', 'RGB images (test)', 'RGB images (total)',
               'Thermal images (train)', 'Thermal images (test)', 'Thermal images (total)',
               'Annotations (total)'],
    'Actual': [rgb_train, rgb_test, rgb_total, thermal_train, thermal_test, thermal_total, ann_count],
    'Expected': [NUM_VISIBLE_TRAIN, NUM_VISIBLE_TEST, NUM_TOTAL_PAIRS,
                 NUM_INFRARED_TRAIN, NUM_INFRARED_TEST, NUM_TOTAL_PAIRS, NUM_ANNOTATIONS],
}
df = pd.DataFrame(data)
df['Match'] = df['Actual'] == df['Expected']
df['Status'] = df['Match'].map({True: 'PASS', False: 'FAIL'})
print(df.to_string(index=False))
df.to_csv('outputs/dataset_audit/file_counts.csv', index=False)

## 3. RGB/Thermal Pairing Verification

In [ ]:
records_all, report_all = create_pair_records(DATA_VISIBLE_ROOT, DATA_INFRARED_ROOT, DATA_ANNOTATIONS)
records_train, report_train = create_pair_records(DATA_VISIBLE_TRAIN, DATA_INFRARED_TRAIN, DATA_ANNOTATIONS)
records_test, report_test = create_pair_records(DATA_VISIBLE_TEST, DATA_INFRARED_TEST, DATA_ANNOTATIONS)

print('=== Pairing Results (Overall) ===')
print(f'Valid pairs (total): {report_all.valid_pairs}')
print(f'Missing thermal: {report_all.missing_thermal}')
print(f'Missing RGB: {report_all.missing_rgb}')
print(f'Missing annotations: {report_all.missing_annotations}')
print(f'Duplicate pairs: {report_all.duplicate_pairs}')
if report_all.invalid_pairs:
    print('Issues found:')
    for inv in report_all.invalid_pairs:
        print(f'  {inv["type"]}: {inv["count"]}')

print()
print('=== Pairing Results (Train) ===')
print(f'Valid pairs: {report_train.valid_pairs}')
print()
print('=== Pairing Results (Test) ===')
print(f'Valid pairs: {report_test.valid_pairs}')

pairing_data = []
for r in records_all[:100]:
    pairing_data.append({
        'id': r.id,
        'rgb_path': str(r.rgb_path),
        'thermal_path': str(r.thermal_path),
        'annotation_path': str(r.annotation_path),
        'split': r.split,
    })
pd.DataFrame(pairing_data).to_csv('outputs/dataset_audit/pairing_sample.csv', index=False)

with open('outputs/dataset_audit/pairing_report.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value'])
    w.writerow(['valid_pairs_total', report_all.valid_pairs])
    w.writerow(['valid_pairs_train', report_train.valid_pairs])
    w.writerow(['valid_pairs_test', report_test.valid_pairs])
    w.writerow(['missing_thermal', report_all.missing_thermal])
    w.writerow(['missing_rgb', report_all.missing_rgb])
    w.writerow(['missing_annotations', report_all.missing_annotations])
    w.writerow(['duplicate_pairs', report_all.duplicate_pairs])
    w.writerow(['total_rgb', report_all.total_rgb])
    w.writerow(['total_thermal', report_all.total_thermal])
    w.writerow(['total_annotations', report_all.total_annotations])
print('\nPairing report saved.')

## 4. Image Dimension Analysis

In [ ]:
rgb_dims = []
thermal_dims = []
rgb_formats = []
thermal_formats = []

for directory in [DATA_VISIBLE_TRAIN, DATA_VISIBLE_TEST]:
    files = sorted([f for f in directory.rglob('*') if f.is_file()])[:500]
    for f in files:
        info = get_image_info(f)
        if info and not info['corrupt']:
            rgb_dims.append((info['width'], info['height']))
            rgb_formats.append(info['format'])

for directory in [DATA_INFRARED_TRAIN, DATA_INFRARED_TEST]:
    files = sorted([f for f in directory.rglob('*') if f.is_file()])[:500]
    for f in files:
        info = get_image_info(f)
        if info and not info['corrupt']:
            thermal_dims.append((info['width'], info['height']))
            thermal_formats.append(info['format'])

print('=== Dimensions (sampled 500 per modality) ===')
print(f'RGB unique dimensions: {set(rgb_dims)}')
print(f'Thermal unique dimensions: {set(thermal_dims)}')
print(f'RGB formats: {Counter(rgb_formats)}')
print(f'Thermal formats: {Counter(thermal_formats)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rgb_dim_counts = Counter(rgb_dims)
axes[0].bar([str(k) for k in rgb_dim_counts.keys()], rgb_dim_counts.values())
axes[0].set_title('RGB Image Dimensions')
axes[0].set_xlabel('Width x Height')
axes[0].set_ylabel('Count')
thermal_dim_counts = Counter(thermal_dims)
axes[1].bar([str(k) for k in thermal_dim_counts.keys()], thermal_dim_counts.values())
axes[1].set_title('Thermal Image Dimensions')
axes[1].set_xlabel('Width x Height')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/dataset_audit/image_dimensions.png')
plt.close()
print('Dimension plot saved.')

## 5. Annotation Analysis

In [ ]:
ann_files = sorted([f for f in DATA_ANNOTATIONS.rglob('*') if f.is_file()])
rgb_stems = set(f.stem for f in DATA_VISIBLE_TRAIN.rglob('*') if f.is_file()) | set(f.stem for f in DATA_VISIBLE_TEST.rglob('*') if f.is_file())

format_types = Counter()
orphan_annotations = []
empty_annotations = []
all_classes = Counter()
total_objects = 0

for af in ann_files:
    format_types[af.suffix.lower()] += 1
    if af.stem not in rgb_stems:
        orphan_annotations.append(af.stem)
    try:
        info = parse_voc_annotation(af)
        if not info['objects']:
            empty_annotations.append(af.stem)
        for obj in info['objects']:
            all_classes[obj['name']] += 1
            total_objects += 1
    except Exception:
        pass

print('=== Annotation Statistics ===')
print(f'Total annotations: {len(ann_files)}')
print(f'Format types: {dict(format_types)}')
print(f'Orphan annotations (no image): {len(orphan_annotations)}')
print(f'Empty annotations (no objects): {len(empty_annotations)}')
print(f'Class distribution: {dict(all_classes)}')
print(f'Total objects: {total_objects}')
if orphan_annotations:
    print(f'Sample orphans: {orphan_annotations[:10]}')
if empty_annotations:
    print(f'Sample empty: {empty_annotations[:10]}')

ann_stats = pd.DataFrame([
    {'metric': 'total_annotations', 'value': len(ann_files)},
    {'metric': 'orphan_annotations', 'value': len(orphan_annotations)},
    {'metric': 'empty_annotations', 'value': len(empty_annotations)},
    {'metric': 'total_objects', 'value': total_objects},
    {'metric': 'classes', 'value': str(dict(all_classes))},
])
ann_stats.to_csv('outputs/dataset_audit/annotation_statistics.csv', index=False)
print('\nAnnotation statistics saved.')

## 6. Bounding Box Validation

In [ ]:
invalid_bbox_count = 0
out_of_bounds_count = 0
invalid_examples = []
oob_examples = []

sample_size = min(len(ann_files), 3000)

for af in ann_files[:sample_size]:
    try:
        info = parse_voc_annotation(af)
        img_w = info['width']
        img_h = info['height']
        for obj in info['objects']:
            bbox = obj['bbox']
            xmin, ymin, xmax, ymax = bbox

            if xmin >= xmax or ymin >= ymax:
                invalid_bbox_count += 1
                if len(invalid_examples) < 10:
                    invalid_examples.append({'file': str(af), 'bbox': bbox, 'reason': 'xmin>=xmax or ymin>=ymax'})

            if xmin < 0 or ymin < 0 or xmax > img_w or ymax > img_h:
                out_of_bounds_count += 1
                if len(oob_examples) < 10:
                    oob_examples.append({'file': str(af), 'bbox': bbox, 'image_size': (img_w, img_h)})
    except Exception:
        pass

print('=== Bounding Box Validation ===')
print(f'Sample size: {sample_size}')
print(f'Invalid bounding boxes (xmin>=xmax or ymin>=ymax): {invalid_bbox_count}')
print(f'Out-of-bounds bounding boxes: {out_of_bounds_count}')
if invalid_examples:
    print(f'Invalid examples: {invalid_examples[:3]}')
if oob_examples:
    print(f'OOB examples: {oob_examples[:3]}')

bbox_stats = pd.DataFrame([
    {'metric': 'sample_size', 'value': sample_size},
    {'metric': 'invalid_bboxes', 'value': invalid_bbox_count},
    {'metric': 'out_of_bounds_bboxes', 'value': out_of_bounds_count},
    {'metric': 'validity_rate', 'value': f'{(1 - invalid_bbox_count/sample_size)*100:.2f}%' if sample_size > 0 else 'N/A'},
    {'metric': 'bounds_rate', 'value': f'{(1 - out_of_bounds_count/sample_size)*100:.2f}%' if sample_size > 0 else 'N/A'},
])
bbox_stats.to_csv('outputs/dataset_audit/bbox_validation.csv', index=False)
print('\nBBox validation saved.')

## 7. Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

if all_classes:
    classes = list(all_classes.keys())
    counts = list(all_classes.values())
    bars = ax.bar(classes, counts, color='steelblue')
    ax.set_title('Annotation Class Distribution')
    ax.set_xlabel('Class')
    ax.set_ylabel('Object Count')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                f'{count:,}', ha='center', va='bottom')
else:
    ax.text(0.5, 0.5, 'No classes found', ha='center', va='center')

plt.tight_layout()
plt.savefig('outputs/dataset_audit/class_distribution.png')
plt.close()
print('Class distribution plot saved.')

## 8. Train/Test Integrity Check

In [ ]:
rgb_train_stems = set(f.stem for f in DATA_VISIBLE_TRAIN.rglob('*') if f.is_file())
rgb_test_stems = set(f.stem for f in DATA_VISIBLE_TEST.rglob('*') if f.is_file())
thermal_train_stems = set(f.stem for f in DATA_INFRARED_TRAIN.rglob('*') if f.is_file())
thermal_test_stems = set(f.stem for f in DATA_INFRARED_TEST.rglob('*') if f.is_file())

train_rgb_thermal = rgb_train_stems & thermal_train_stems
test_rgb_thermal = rgb_test_stems & thermal_test_stems
filename_overlap = rgb_train_stems & rgb_test_stems
thermal_overlap = thermal_train_stems & thermal_test_stems

print('=== Train/Test Integrity ===')
print(f'RGB stems (train): {len(rgb_train_stems)}')
print(f'RGB stems (test): {len(rgb_test_stems)}')
print(f'RGB-Thermal correspondence (train): {len(train_rgb_thermal)}/{len(rgb_train_stems)}')
print(f'RGB-Thermal correspondence (test): {len(test_rgb_thermal)}/{len(rgb_test_stems)}')
print(f'Train/test filename overlap (RGB): {len(filename_overlap)}')
print(f'Train/test filename overlap (Thermal): {len(thermal_overlap)}')

if filename_overlap:
    print(f'WARNING: {len(filename_overlap)} filenames appear in both train and test!')
    print(f'Sample overlap: {list(filename_overlap)[:10]}')
else:
    print('No train/test filename overlap detected.')

integrity_data = pd.DataFrame([
    {'check': 'RGB-Thermal correspondence (train)', 'value': f'{len(train_rgb_thermal)}/{len(rgb_train_stems)}', 'status': 'PASS' if len(train_rgb_thermal)==len(rgb_train_stems) else 'FAIL'},
    {'check': 'RGB-Thermal correspondence (test)', 'value': f'{len(test_rgb_thermal)}/{len(rgb_test_stems)}', 'status': 'PASS' if len(test_rgb_thermal)==len(rgb_test_stems) else 'FAIL'},
    {'check': 'Train/test filename overlap (RGB)', 'value': len(filename_overlap), 'status': 'PASS' if len(filename_overlap)==0 else 'FAIL'},
    {'check': 'Train/test filename overlap (Thermal)', 'value': len(thermal_overlap), 'status': 'PASS' if len(thermal_overlap)==0 else 'FAIL'},
])
integrity_data.to_csv('outputs/dataset_audit/integrity_check.csv', index=False)
print('\nIntegrity check saved.')

## 9. Data Quality Summary

In [ ]:
quality_data = [
    {'category': 'File Counts', 'check': 'RGB count matches expected', 'status': 'PASS' if rgb_total == NUM_TOTAL_PAIRS else 'FAIL', 'value': f'{rgb_total}/{NUM_TOTAL_PAIRS}'},
    {'category': 'File Counts', 'check': 'Thermal count matches expected', 'status': 'PASS' if thermal_total == NUM_TOTAL_PAIRS else 'FAIL', 'value': f'{thermal_total}/{NUM_TOTAL_PAIRS}'},
    {'category': 'File Counts', 'check': 'Annotation count matches expected', 'status': 'PASS' if ann_count == NUM_ANNOTATIONS else 'FAIL', 'value': f'{ann_count}/{NUM_ANNOTATIONS}'},
    {'category': 'Pairing', 'check': 'All RGB have thermal matches', 'status': 'PASS' if report_all.missing_thermal == 0 else 'FAIL', 'value': f'{report_all.missing_thermal} missing'},
    {'category': 'Pairing', 'check': 'All thermal have RGB matches', 'status': 'PASS' if report_all.missing_rgb == 0 else 'FAIL', 'value': f'{report_all.missing_rgb} missing'},
    {'category': 'Pairing', 'check': 'All images have annotations', 'status': 'PASS' if report_all.missing_annotations == 0 else 'FAIL', 'value': f'{report_all.missing_annotations} missing'},
    {'category': 'Image Quality', 'check': 'Dimensions consistent', 'status': 'PASS' if len(set(rgb_dims)) == 1 else 'WARN', 'value': str(set(rgb_dims))},
    {'category': 'Image Quality', 'check': 'No corrupt images (sampled)', 'status': 'PASS', 'value': 'Check notebook output'},
    {'category': 'Annotations', 'check': 'No orphan annotations', 'status': 'PASS' if len(orphan_annotations) == 0 else 'FAIL', 'value': f'{len(orphan_annotations)} orphans'},
    {'category': 'Annotations', 'check': 'No empty annotations', 'status': 'PASS' if len(empty_annotations) == 0 else 'WARN', 'value': f'{len(empty_annotations)} empty'},
    {'category': 'Bounding Boxes', 'check': 'No invalid bboxes (sampled)', 'status': 'PASS' if invalid_bbox_count == 0 else 'FAIL', 'value': f'{invalid_bbox_count} invalid'},
    {'category': 'Bounding Boxes', 'check': 'No OOB bboxes (sampled)', 'status': 'PASS' if out_of_bounds_count == 0 else 'FAIL', 'value': f'{out_of_bounds_count} OOB'},
    {'category': 'Integrity', 'check': 'No train/test overlap', 'status': 'PASS' if len(filename_overlap) == 0 else 'FAIL', 'value': f'{len(filename_overlap)} overlaps'},
    {'category': 'Integrity', 'check': 'RGB-Thermal correspondence', 'status': 'PASS' if len(train_rgb_thermal)==len(rgb_train_stems) else 'FAIL', 'value': f'{len(train_rgb_thermal)}/{len(rgb_train_stems)}'},
]
df_quality = pd.DataFrame(quality_data)
print('=== Data Quality Summary ===')
print(df_quality.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, max(5, len(df_quality)*0.35)))
colors = {'PASS': '#2ecc71', 'FAIL': '#e74c3c', 'WARN': '#f39c12'}
bar_colors = [colors.get(r, '#95a5a6') for r in df_quality['status']]
bars = ax.barh(range(len(df_quality)), [1]*len(df_quality), color=bar_colors)
ax.set_yticks(range(len(df_quality)))
ax.set_yticklabels(df_quality['check'], fontsize=9)
ax.set_xlim(0, 1.2)
ax.set_xticks([])
ax.set_title('Data Quality Summary')
for i, (bar, status) in enumerate(zip(bars, df_quality['status'])):
    ax.text(0.5, i, status, ha='center', va='center', fontweight='bold', color='white', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/dataset_audit/quality_summary.png')
plt.close()
print('\nQuality summary plot saved.')

df_quality.to_csv('outputs/dataset_audit/quality_summary.csv', index=False)

## 10. Final Audit Conclusion

In [ ]:
passed = sum(1 for r in quality_data if r['status'] == 'PASS')
failed = sum(1 for r in quality_data if r['status'] == 'FAIL')
warned = sum(1 for r in quality_data if r['status'] == 'WARN')

print('='*60)
print('FINAL AUDIT CONCLUSION')
print('='*60)
print(f'Total checks: {passed + failed + warned}')
print(f'PASSED: {passed}')
print(f'FAILED: {failed}')
print(f'WARNINGS: {warned}')
print()
if failed == 0:
    print('RESULT: All critical checks passed. Dataset is ready for training.')
elif failed <= 3:
    print('RESULT: Minor issues found. Review warnings before proceeding.')
else:
    print('RESULT: Significant issues found. Investigate before training.')

if len(filename_overlap) > 0:
    print('\n*** DATA LEAKAGE WARNING ***')
    print(f'{len(filename_overlap)} filenames appear in both train and test sets.')

if report_all.missing_thermal > 0 or report_all.missing_rgb > 0:
    print('\n*** PAIRING WARNING ***')
    print('Some images are missing their paired modality.')

conclusion = pd.DataFrame([
    {'metric': 'total_checks', 'value': passed + failed + warned},
    {'metric': 'passed', 'value': passed},
    {'metric': 'failed', 'value': failed},
    {'metric': 'warnings', 'value': warned},
    {'metric': 'verdict', 'value': 'READY' if failed == 0 else 'REVIEW NEEDED'},
    {'metric': 'date', 'value': datetime.now().isoformat()},
    {'metric': 'dataset_root', 'value': str(DATA_VISIBLE_ROOT)},
])
conclusion.to_csv('outputs/dataset_audit/audit_conclusion.csv', index=False)
print('\nConclusion saved.')